# Feature Scaling

**Topic:** Data Preprocessing

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, VBox, HBox, HTML
from IPython.display import display, clear_output

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

import seaborn as sns
titanic = sns.load_dataset("titanic")

from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** why numeric features on different scales give some features an unfair advantage in a model
- **Explain** the difference between MinMax scaling, Standard scaling, and Robust scaling
- **Interpret** how the presence of outliers affects which scaler you should choose

> **Tip:** Look at the unscaled box plots first. Notice how `fare` completely dominates the axis. Then switch to Standard Scaler and watch all four features become comparable.

---
## How we got here

In `03_encoding_categorical_variables.ipynb` we converted text categories into numeric columns. Now all our features are numbers, but they live on wildly different scales.

The Titanic's `fare` column ranges from 0 to 512. `sibsp` (siblings and spouses) only goes from 0 to 8. `age` sits between 0 and 80. That gap is a problem for any algorithm that measures distance or follows gradients.

This connects directly to two earlier concepts: `statistics/05_dispersion.ipynb` introduced standard deviation and variance, the building blocks of Standard scaling. `math/linear_algebra/06_norms_and_distances.ipynb` showed how distance is calculated in multiple dimensions: exactly the operation distance-based algorithms use when unscaled features mislead them.

---
## Why this matters for data science

Any algorithm that uses distance or gradient descent treats large-scale features as more important by default. A column measured in dollars and a column measured in years will be weighted as if dollars matter 100 times more than years, purely because of their units. The model has no way to know those units are arbitrary.

---
## Try it yourself

In [2]:
out_box  = Output()
out_dist = Output()

NUMERIC_COLS = ["age", "fare", "sibsp", "parch"]
df_num = titanic[NUMERIC_COLS].dropna().copy()

SCALERS = {
    "None (raw data)": None,
    "MinMax Scaler":   MinMaxScaler(),
    "Standard Scaler": StandardScaler(),
    "Robust Scaler":   RobustScaler(),
}

scale_dd = Dropdown(
    options=list(SCALERS.keys()),
    value="None (raw data)",
    description="Scaler:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

COL_COLORS = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["muted"]]

def apply_scaler(df, method):
    scaler = SCALERS[method]
    if scaler is None:
        return df.copy()
    return pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

def render(change=None):
    method = scale_dd.value
    scaled = apply_scaler(df_num, method)

    box_traces = [
        go.Box(y=scaled[col], name=col, marker_color=COL_COLORS[i], showlegend=True)
        for i, col in enumerate(NUMERIC_COLS)
    ]
    layout_box = base_layout(
        title=f"Titanic Numeric Features: {method}",
        xaxis_title="Feature",
        yaxis_title="Value",
    )

    with out_box:
        clear_output(wait=True)
        display(go.FigureWidget(data=box_traces, layout=layout_box))

    dist_traces = [go.Histogram(
        x=scaled["fare"], nbinsx=35,
        marker_color=PALETTE["primary"], opacity=0.8,
        name="fare",
    )]
    layout_dist = base_layout(
        title=f"Fare Distribution: {method} (outliers visible)",
        xaxis_title="Scaled value",
        yaxis_title="Count",
    )
    with out_dist:
        clear_output(wait=True)
        display(go.FigureWidget(data=dist_traces, layout=layout_dist))

scale_dd.observe(render, names="value")
display(VBox([scale_dd, out_box, out_dist]))
render()

---
## What's happening?

Feature scaling transforms all numeric columns so they live on a comparable scale. Critically, it does not change the *information* in the data; it changes the *units*.

Think of it this way: if you measure height in centimeters (150–190) and weight in grams (50,000–100,000), no algorithm should assume weight matters 1,000 times more than height just because of the units. Scaling removes that unit bias.

**MinMax Scaler** compresses each feature into the range [0, 1] by subtracting the minimum and dividing by the range. It preserves the exact shape of the distribution but is sensitive to outliers: a single extreme fare value sets the ceiling for everyone else.

**Standard Scaler** subtracts the mean and divides by the standard deviation. The result has mean=0 and std=1. It does not force a specific range but handles outliers better than MinMax because the standard deviation is less sensitive to extremes than the range.

**Robust Scaler** uses the median and interquartile range instead of mean and standard deviation. It is the best choice when your data has significant outliers, like the `fare` column in Titanic is a good example.

| Scaler | Formula | Output range | Best used when | sklearn class |
|---|---|---|---|---|
| MinMax | (x − min) / (max − min) | [0, 1] | Data has no significant outliers | `MinMaxScaler` |
| Standard | (x − mean) / std | Centered at 0 | General-purpose; most common choice | `StandardScaler` |
| Robust | (x − median) / IQR | Centered at 0, outlier-resistant | Data has significant outliers | `RobustScaler` |

---
## When you need this step

Algorithms that **measure distances** between data points, or that **optimize by taking iterative steps**, are sensitive to scale:

- If two features have very different ranges, the one with larger values will dominate distance calculations, not because it is more important, but because it is numerically larger.
- Algorithms that adjust their internal weights by following a gradient will converge slowly (or not at all) when features are on very different scales.

**Feature scaling is required for these algorithm families.**

## When you can skip it

Algorithms that **split data on thresholds** (asking "is this value above or below a cut point?") are completely indifferent to scale. Whether `fare` is 100 dollars or 1.2 on a standard scale, the split point adjusts accordingly. These algorithms do not need feature scaling.

**Coming up soon:** The first algorithm you will meet, linear regression, optimizes iteratively. Scaling will matter. When you reach that notebook, you will see exactly how unscaled features slow down training.

---
## Key takeaway

> **Scaling does not change the information in your data; it changes the units, so every feature gets a fair hearing from the model. Whether you need it depends on which algorithm you use.**

---
*Next up: 05_feature_scaling_in_practice.ipynb, watching scaling change a real decision boundary on the Titanic data*